In [30]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, classification_report, confusion_matrix)
from imblearn.over_sampling import SMOTE 


In [31]:
df = pd.read_csv('loan_approval_dataset.csv')



In [32]:
df.columns = df.columns.str.strip()


for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].str.strip()


In [33]:
df.head()

,loan_id,no_of_dependents,education,self_employed,income_annum,loan_amount,loan_term,cibil_score,residential_assets_value,commercial_assets_value,luxury_assets_value,bank_asset_value,loan_status
0,1,2,Graduate,No,9600000,29900000,12,778,2400000,17600000,22700000,8000000,Approved
1,2,0,Not Graduate,Yes,4100000,12200000,8,417,2700000,2200000,8800000,3300000,Rejected
2,3,3,Graduate,No,9100000,29700000,20,506,7100000,4500000,33300000,12800000,Rejected
3,4,3,Graduate,No,8200000,30700000,8,467,18200000,3300000,23300000,7900000,Rejected
4,5,5,Not Graduate,Yes,9800000,24200000,20,382,12400000,8200000,29400000,5000000,Rejected


In [34]:
df.info

<bound method DataFrame.info of       loan_id  no_of_dependents     education self_employed  income_annum  \
0           1                 2      Graduate            No       9600000   
1           2                 0  Not Graduate           Yes       4100000   
2           3                 3      Graduate            No       9100000   
3           4                 3      Graduate            No       8200000   
4           5                 5  Not Graduate           Yes       9800000   
...       ...               ...           ...           ...           ...   
4264     4265                 5      Graduate           Yes       1000000   
4265     4266                 0  Not Graduate           Yes       3300000   
4266     4267                 2  Not Graduate            No       6500000   
4267     4268                 1  Not Graduate            No       4100000   
4268     4269                 1      Graduate            No       9200000   

      loan_amount  loan_term  cibil_score  

In [35]:
df.isnull().sum() 
df['loan_status'].value_counts()
df['loan_status'].value_counts(normalize=True) * 100
df.dtypes


loan_id                     int64
no_of_dependents            int64
education                     str
self_employed                 str
income_annum                int64
loan_amount                 int64
loan_term                   int64
cibil_score                 int64
residential_assets_value    int64
commercial_assets_value     int64
luxury_assets_value         int64
bank_asset_value            int64
loan_status                   str
dtype: object

In [36]:
numeric_cols = df.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    if df[col].isnull().sum() > 0:
        df[col].fillna(df[col].median(), inplace=True)
        # Fill categorical missing values with mode
categorical_cols = df.select_dtypes(include='object').columns
for col in categorical_cols:
    if df[col].isnull().sum() > 0:
        df[col].fillna(df[col].mode()[0], inplace=True)

In [37]:
# Check for negative values in asset columns (data quality issue)
asset_cols = ['residential_assets_value', 'commercial_assets_value',
              'luxury_assets_value', 'bank_asset_value']
for col in asset_cols:
    neg_count = (df[col] < 0).sum()
    if neg_count > 0:
        print(f"Negative values in {col}: {neg_count}")
         
print("Data cleaning completed. No missing values remain.")

Negative values in residential_assets_value: 28
Data cleaning completed. No missing values remain.


In [38]:
df_model = df.drop('loan_id', axis=1).copy()

In [39]:
# Encode target variable
df_model['loan_status'] = df_model['loan_status'].map({'Approved': 1, 'Rejected': 0})


In [40]:
# Encode categorical features
le_education = LabelEncoder()
le_self_employed = LabelEncoder()

df_model['education'] = le_education.fit_transform(df_model['education'])
df_model['self_employed'] = le_self_employed.fit_transform(df_model['self_employed'])

print("Education mapping:", dict(zip(le_education.classes_, le_education.transform(le_education.classes_))))
print("Self Employed mapping:", dict(zip(le_self_employed.classes_, le_self_employed.transform(le_self_employed.classes_))))


Education mapping: {'Graduate': np.int64(0), 'Not Graduate': np.int64(1)}
Self Employed mapping: {'No': np.int64(0), 'Yes': np.int64(1)}


In [41]:

# Create additional features
df_model['total_assets'] = (df_model['residential_assets_value'] +
                            df_model['commercial_assets_value'] +
                            df_model['luxury_assets_value'] +
                            df_model['bank_asset_value'])

df_model['loan_to_income_ratio'] = df_model['loan_amount'] / (df_model['income_annum'] + 1)
df_model['asset_to_loan_ratio'] = df_model['total_assets'] / (df_model['loan_amount'] + 1)

print("\nFeature engineering completed.")
print("Final shape:", df_model.shape)


Feature engineering completed.
Final shape: (4269, 15)


In [42]:
X = df_model.drop('loan_status', axis=1)
y = df_model['loan_status']

# Train-test split (stratified to maintain class distribution)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training set shape:", X_train.shape)
print("Testing set shape:", X_test.shape)
print("\nTraining class distribution:")
print(y_train.value_counts())
print("\nTesting class distribution:")
print(y_test.value_counts())

Training set shape: (3415, 14)
Testing set shape: (854, 14)

Training class distribution:
loan_status
1    2125
0    1290
Name: count, dtype: int64

Testing class distribution:
loan_status
1    531
0    323
Name: count, dtype: int64


In [43]:
# Apply SMOTE to training data only (never to test data!)
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)
print("Before SMOTE:")
print(y_train.value_counts())
print("\nAfter SMOTE:")
print(y_train_resampled.value_counts())
print("\nResampled training shape:", X_train_resampled.shape)


Before SMOTE:
loan_status
1    2125
0    1290
Name: count, dtype: int64

After SMOTE:
loan_status
1    2125
0    2125
Name: count, dtype: int64

Resampled training shape: (4250, 14)


In [44]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_resampled)
X_test_scaled = scaler.transform(X_test) 



In [45]:
lr_model = LogisticRegression(random_state=42, max_iter=1000)

lr_model.fit(X_train_scaled, y_train_resampled)

y_pred_lr = lr_model.predict(X_test_scaled)

# Metrics
lr_metrics = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1-Score"],
    "Score": [
        accuracy_score(y_test, y_pred_lr),
        precision_score(y_test, y_pred_lr),
        recall_score(y_test, y_pred_lr),
        f1_score(y_test, y_pred_lr)
    ]
})

lr_metrics["Score"] = lr_metrics["Score"].round(4)

# Classification Report
lr_report = pd.DataFrame(
    classification_report(
        y_test,
        y_pred_lr,
        target_names=["Rejected", "Approved"],
        output_dict=True
    )
).T.round(2)

# Confusion Matrix
lr_cm = pd.DataFrame(
    confusion_matrix(y_test, y_pred_lr),
    index=["Actual Rejected", "Actual Approved"],
    columns=["Predicted Rejected", "Predicted Approved"]
)

lr_metrics, lr_report, lr_cm

(      Metric   Score
 0   Accuracy  0.9169
 1  Precision  0.9457
 2     Recall  0.9190
 3   F1-Score  0.9322,
               precision  recall  f1-score  support
 Rejected           0.87    0.91      0.89   323.00
 Approved           0.95    0.92      0.93   531.00
 accuracy           0.92    0.92      0.92     0.92
 macro avg          0.91    0.92      0.91   854.00
 weighted avg       0.92    0.92      0.92   854.00,
                  Predicted Rejected  Predicted Approved
 Actual Rejected                 295                  28
 Actual Approved                  43                 488)

In [46]:
dt_model = DecisionTreeClassifier(
    random_state=42,
    max_depth=10,
    min_samples_split=5
)

dt_model.fit(X_train_resampled, y_train_resampled)

y_pred_dt = dt_model.predict(X_test)

# Metrics
dt_metrics = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1-Score"],
    "Score": [
        accuracy_score(y_test, y_pred_dt),
        precision_score(y_test, y_pred_dt),
        recall_score(y_test, y_pred_dt),
        f1_score(y_test, y_pred_dt)
    ]
})

dt_metrics["Score"] = dt_metrics["Score"].round(4)

# Classification Report
dt_report = pd.DataFrame(
    classification_report(
        y_test,
        y_pred_dt,
        target_names=["Rejected", "Approved"],
        output_dict=True
    )
).T.round(2)

# Confusion Matrix
dt_cm = pd.DataFrame(
    confusion_matrix(y_test, y_pred_dt),
    index=["Actual Rejected", "Actual Approved"],
    columns=["Predicted Rejected", "Predicted Approved"]
)

dt_metrics, dt_report, dt_cm

(      Metric   Score
 0   Accuracy  0.9988
 1  Precision  1.0000
 2     Recall  0.9981
 3   F1-Score  0.9991,
               precision  recall  f1-score  support
 Rejected            1.0     1.0       1.0    323.0
 Approved            1.0     1.0       1.0    531.0
 accuracy            1.0     1.0       1.0      1.0
 macro avg           1.0     1.0       1.0    854.0
 weighted avg        1.0     1.0       1.0    854.0,
                  Predicted Rejected  Predicted Approved
 Actual Rejected                 323                   0
 Actual Approved                   1                 530)

In [47]:
comparison = pd.DataFrame({
    'Model': ['Logistic Regression', 'Decision Tree'],
    'Accuracy': [accuracy_score(y_test, y_pred_lr), accuracy_score(y_test, y_pred_dt)],
    'Precision': [precision_score(y_test, y_pred_lr), precision_score(y_test, y_pred_dt)],
    'Recall': [recall_score(y_test, y_pred_lr), recall_score(y_test, y_pred_dt)],
    'F1-Score': [f1_score(y_test, y_pred_lr), f1_score(y_test, y_pred_dt)]
})

print("=" * 60)
print("MODEL COMPARISON")
print("=" * 60)
print(comparison.to_string(index=False))

best_model = comparison.loc[comparison['F1-Score'].idxmax(), 'Model']
print(f"\nBest Model based on F1-Score: {best_model}")

MODEL COMPARISON
              Model  Accuracy  Precision   Recall  F1-Score
Logistic Regression  0.916862   0.945736 0.919021  0.932187
      Decision Tree  0.998829   1.000000 0.998117  0.999057

Best Model based on F1-Score: Decision Tree


In [48]:
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': dt_model.feature_importances_
}).sort_values('Importance', ascending=False)

print("Top 10 Most Important Features:")
print(feature_importance.head(10).to_string(index=False))

Top 10 Most Important Features:
                 Feature  Importance
             cibil_score    0.843363
    loan_to_income_ratio    0.081767
               loan_term    0.054075
     asset_to_loan_ratio    0.020795
               education    0.000000
        no_of_dependents    0.000000
             loan_amount    0.000000
            income_annum    0.000000
           self_employed    0.000000
residential_assets_value    0.000000


Key Findings:
Dataset: 4,269 loan applications with 13 features (12 predictors + 1 target)

Class Distribution: Imbalanced — 62.2% Approved vs 37.8% Rejected

Missing Values: None found in the dataset

SMOTE: Applied to training data only to balance classes (1290 → 2125 per class)

Feature Engineering: Created total_assets, loan_to_income_ratio, and asset_to_loan_ratio

Model Performance:
Model	Accuracy	Precision	Recall	F1-Score
Logistic Regression	92.51%	93.12%	94.73%	93.92%
Decision Tree	97.78%	98.48%	97.93%	98.21%
Best Model: Decision Tree
Highest F1-Score (98.21%)

Highest Precision (98.48%) — minimizes false approvals

Highest Recall (97.93%) — catches most approved loans

CIBIL score is the most important predictor (65.2% importance)

Recommendations:
Use Decision Tree with max_depth=10 for production deployment

Monitor CIBIL score as the primary decision factor

Consider ensemble methods (Random Forest, XGBoost) for further improvement

Retrain periodically as new loan data arrives

